# T12 CNN Woodtype / Structure Soft Routing

species???????woodtype ? wood_structure ?softmax?????????????????????


In [ ]:
from getpass import getpass
from pathlib import Path
import os
import subprocess
import sys

GITHUB_REPO = '2Kentaro1/wood-moisture-2d-cnn'
PROJECT_DIR = Path('/content/wood-moisture-2d-cnn')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/wood-moisture-2d-cnn-outputs')
REPO_URL = f'https://github.com/{GITHUB_REPO}.git'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    os.environ['OUTPUT_DIR'] = str(DRIVE_OUTPUT_DIR)
except Exception as exc:
    print(f'Google Drive mount skipped: {exc}')
    os.environ.setdefault('OUTPUT_DIR', 'outputs')

token = os.environ.get('GITHUB_TOKEN', '').strip()
if not token:
    token = getpass('GitHub token (empty for public repo): ').strip()
clone_url = REPO_URL if not token else f'https://{token}@github.com/{GITHUB_REPO}.git'

if PROJECT_DIR.exists() and (PROJECT_DIR / '.git').exists():
    os.chdir(PROJECT_DIR)
    try:
        if token:
            subprocess.run(['git', 'pull', clone_url, 'main'], check=True)
            subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)
        else:
            subprocess.run(['git', 'pull'], check=True)
    except subprocess.CalledProcessError as exc:
        print(f'git pull skipped/failed: {exc}')
else:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    if PROJECT_DIR.exists() and not (PROJECT_DIR / '.git').exists():
        raise RuntimeError(f'{PROJECT_DIR} exists but is not a git repository. Rename/remove it or set PROJECT_DIR.')
    subprocess.run(['git', 'clone', clone_url, str(PROJECT_DIR)], check=True)
    os.chdir(PROJECT_DIR)
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO_URL], check=False)

os.environ['PROJECT_ROOT'] = str(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
Path(os.environ['OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT={os.environ["PROJECT_ROOT"]}')
print(f'OUTPUT_DIR={os.environ["OUTPUT_DIR"]}')
print('src exists:', (PROJECT_DIR / 'src').exists())
print('train exists:', (PROJECT_DIR / 'data' / 'train.csv').exists())
print('test exists:', (PROJECT_DIR / 'data' / 'test.csv').exists())

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)


In [ ]:
!apt-get -qq update >/dev/null
!apt-get -qq install -y fonts-noto-cjk >/dev/null
from pathlib import Path
import os
T12_OUTPUT_DIR = Path('/content/drive/MyDrive/wood-moisture-2d-cnn-outputs/T12_cnn_soft_routing')
os.environ['T12_OUTPUT_DIR'] = str(T12_OUTPUT_DIR)
print('PROJECT_ROOT =', Path.cwd())
print('OUTPUT_DIR =', os.environ.get('OUTPUT_DIR'))
print('T12_OUTPUT_DIR =', T12_OUTPUT_DIR)
print('data/train.csv exists =', Path('data/train.csv').exists())
print('data/test.csv exists =', Path('data/test.csv').exists())


## Train Woodtype / Wood Structure CNNs


In [ ]:
!python -m src.training.train_soft_routing --tasks woodtype wood_structure --output-dir "$T12_OUTPUT_DIR" --epochs 80 --batch-size 32


## OOF / Test Probability Check


In [ ]:
import pandas as pd
for path in [
    T12_OUTPUT_DIR / 'oof/train_woodtype_oof_probs.csv',
    T12_OUTPUT_DIR / 'test/test_woodtype_probs.csv',
    T12_OUTPUT_DIR / 'oof/train_wood_structure_oof_probs.csv',
    T12_OUTPUT_DIR / 'test/test_wood_structure_probs.csv',
]:
    print('\n', path)
    display(pd.read_csv(path).head())


## Metrics / Probability Diagnostics


In [ ]:
import json
for task in ['woodtype', 'wood_structure']:
    metrics_path = T12_OUTPUT_DIR / 'metrics' / f'{task}_metrics.json'
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print(task, {k: metrics[k] for k in ['accuracy', 'macro_f1', 'weighted_f1', 'logloss']})
    display(pd.read_csv(T12_OUTPUT_DIR / 'metrics' / f'{task}_train_test_probability_stats.csv'))


## Figures


In [ ]:
from IPython.display import Image, display
for task in ['woodtype', 'wood_structure']:
    for name in ['confusion_matrix', 'species_probability_boxplot', 'true_class_probability_violin', 'train_test_probability_comparison', 'uncertainty_train', 'embedding_pca']:
        path = T12_OUTPUT_DIR / 'figures' / f'{task}_{name}.png'
        print(path)
        if path.exists():
            display(Image(filename=str(path)))
        else:
            print('missing')


## Saved Files


In [ ]:
for path in sorted(T12_OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(T12_OUTPUT_DIR))
